# Netflix Recommendation System — GPU Local Test Notebook

> **Hardware target**: RTX 3050 6 GB Laptop GPU + 16 GB system RAM, Windows 11, Python 3.13

## Settings vs Other Versions

| Setting | Kaggle Notebook | **This Notebook** | Crash Version |
|---------|:--------------:|:-----------------:|:-------------:|
| Sample fraction | 10% (~10M) | **5% (~5M)** | 10% |
| NCF device | GPU | **GPU (RTX 3050)** | CPU overload |
| NCF batch size | 2048 | **1024** | 2048 |
| NCF hidden | [128,64,32] | **[128,64,32]** | [128,64,32] |
| NCF epochs | 10 | **10** | 10 |
| DataLoader workers | 2 | **0** (Windows safer) | 2 |
| VRAM guard | ❌ | ✅ | ❌ |
| RAM guard | ❌ | ✅ | ❌ |
| KNN | ✅ | ❌ skipped locally | ✅ |

## Before running
Make sure your PyTorch has CUDA support:
```python
import torch
print(torch.cuda.is_available())   # must be True
print(torch.cuda.get_device_name(0))  # should show RTX 3050
```
If it prints `False`, run in a terminal:
```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
```


## 0  Imports & Memory Utilities

In [ ]:
import os, gc, sys, time, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from contextlib import contextmanager

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    print('[WARN] psutil not installed.  pip install psutil')
    _HAS_PSUTIL = False


def _rss_mb():
    if _HAS_PSUTIL:
        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    return -1.0

def _avail_mb():
    if _HAS_PSUTIL:
        return psutil.virtual_memory().available / 1024**2
    return 99999.0

def mem_mb(obj=None):
    if isinstance(obj, pd.DataFrame):
        return obj.memory_usage(deep=True).sum() / 1024**2
    return _rss_mb()

def vram_mb():
    """GPU VRAM allocated (MB). Returns 0 if CUDA not available."""
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.memory_allocated(0) / 1024**2
    except Exception:
        pass
    return 0.0

@contextmanager
def mem_snapshot(label='block'):
    before_ram  = _rss_mb()
    before_vram = vram_mb()
    t0 = time.time()
    yield
    after_ram  = _rss_mb()
    after_vram = vram_mb()
    elapsed = time.time() - t0
    print(f'[{label}]'
          f'  RAM: {before_ram:.0f}->{after_ram:.0f} MB (delta {after_ram-before_ram:+.0f})'
          f'  VRAM: {before_vram:.0f}->{after_vram:.0f} MB'
          f'  time: {elapsed:.1f}s')

def ram_guard(need_mb=1500, label='next step'):
    avail = _avail_mb()
    if avail < need_mb:
        raise RuntimeError(
            f'RAM guard: [{label}]\n'
            f'  Available {avail:.0f} MB < required {need_mb} MB.\n'
            f'  Close Chrome/other apps and restart the kernel.'
        )
    print(f'RAM guard OK [{label}]: {avail:.0f} MB free')

def vram_guard(need_mb=500, label='next step'):
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            free  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**2
            if free < need_mb:
                raise RuntimeError(
                    f'VRAM guard: [{label}]\n'
                    f'  Free VRAM {free:.0f} MB < required {need_mb} MB.\n'
                    f'  Run: torch.cuda.empty_cache() or restart kernel.'
                )
            print(f'VRAM guard OK [{label}]: {free:.0f} MB free')
    except RuntimeError:
        raise
    except Exception:
        pass

print(f'Python  : {sys.version}')
print(f'OS      : {platform.platform()}')
print(f'Pandas  : {pd.__version__}  NumPy: {np.__version__}')
print(f'RAM RSS : {_rss_mb():.0f} MB  |  Available: {_avail_mb():.0f} MB')


## 1  GPU Detection & PyTorch Check

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_vram = props.total_memory / 1024**2
    free_vram  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**2
    print(f'GPU             : {props.name}')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'VRAM total      : {total_vram:.0f} MB')
    print(f'VRAM free       : {free_vram:.0f} MB')
    DEVICE = torch.device('cuda')
else:
    print()
    print('WARNING: CUDA not available — falling back to CPU.')
    print('To fix: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124')
    DEVICE = torch.device('cpu')

print(f'\nUsing device: {DEVICE}')


## 2  Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DATA_DIR  = 'data/raw'
PROCESSED_DIR = 'data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── Sampling ──────────────────────────────────────────────────────────────────
# 5%  -> ~5M rows -> ~600 MB RAM, comfortable on 16 GB + other apps
# Try 0.10 (10%) only after confirming 5% works without issues
SAMPLE_FRACTION    = 0.05
RAM_LIMIT_MB       = 5000   # hard stop during file parse if RSS exceeds this

RANDOM_SEED        = 42
TEST_SIZE          = 0.20
MIN_USER_RATINGS   = 20
MIN_MOVIE_RATINGS  = 50
RELEVANCE_THRESHOLD = 3.5
TOP_K              = 10

# ── SVD ───────────────────────────────────────────────────────────────────────
SVD_PARAMS = {
    'n_factors': 100,
    'n_epochs':  20,
    'lr_all':    0.005,
    'reg_all':   0.02,
}

# ── NCF — tuned for RTX 3050 6 GB ────────────────────────────────────────────
# VRAM estimate: embeddings ~50 MB + batch ~10 MB + activations ~30 MB = ~90 MB
# Way under the 6 GB limit, so we keep the same architecture as Kaggle.
NCF_PARAMS = {
    'embedding_dim': 64,
    'hidden_layers': [128, 64, 32],
    'dropout':       0.2,
    'lr':            0.001,
    'batch_size':    1024,   # halved vs Kaggle (2048) for laptop thermals
    'epochs':        10,
    'patience':      3,
}

np.random.seed(RANDOM_SEED)
print('Config OK')
print(f'SAMPLE_FRACTION = {SAMPLE_FRACTION} (~{SAMPLE_FRACTION*100:.0f}M rows expected from 100M total)')
print(f'NCF device      = {DEVICE}')
print(f'NCF batch_size  = {NCF_PARAMS["batch_size"]}')
print(f'RAM available   = {_avail_mb():.0f} MB')


## 3  Memory-Optimised Data Loading

Same chunked per-file approach as the Kaggle notebook:
- Parses files one at a time → samples → frees before the next file.
- Uses `int8` ratings and `int32` IDs at parse time.
- Falls back to **synthetic data** if raw files are not found locally.


In [ ]:
PARQUET_PATH = os.path.join(PROCESSED_DIR,
    f'local_sample_{int(SAMPLE_FRACTION*1000):03d}pct.parquet')


def parse_combined_lean(filepath, max_rss=RAM_LIMIT_MB):
    fsize = os.path.getsize(filepath) / 1024**2
    print(f'  Parsing {os.path.basename(filepath)} ({fsize:.0f} MB)...')
    uids, mids, rats, dts = [], [], [], []
    cur = None
    n   = 0
    with open(filepath, 'r') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line: continue
            if line.endswith(':'):
                cur = int(line[:-1])
            else:
                uid, rat, dt = line.split(',')
                uids.append(int(uid)); mids.append(cur)
                rats.append(int(rat)); dts.append(dt)
                n += 1
                if n % 500_000 == 0 and _rss_mb() > max_rss:
                    print(f'    RSS {_rss_mb():.0f} MB > limit {max_rss} MB — stopping at {n:,} rows')
                    break
    df = pd.DataFrame({
        'user_id':  pd.array(uids, dtype='int32'),
        'movie_id': pd.array(mids, dtype='int16'),
        'rating':   pd.array(rats, dtype='int8'),
        'date':     dts,
    })
    del uids, mids, rats, dts; gc.collect()
    print(f'    -> {len(df):,} rows | {mem_mb(df):.0f} MB')
    return df


def load_and_sample(data_dir, frac=SAMPLE_FRACTION, seed=RANDOM_SEED):
    parts = []
    for i in range(1, 5):
        fp = os.path.join(data_dir, f'combined_data_{i}.txt')
        if not os.path.exists(fp): continue
        ram_guard(need_mb=1500, label=f'parse file {i}')
        with mem_snapshot(f'file {i}'):
            full   = parse_combined_lean(fp)
            uids   = full['user_id'].unique()
            rng    = np.random.default_rng(seed + i)
            sel    = rng.choice(uids, size=max(1, int(len(uids)*frac)), replace=False)
            sample = full[full['user_id'].isin(sel)].copy()
            del full; gc.collect()
            print(f'    sampled {len(sample):,} rows | {mem_mb(sample):.0f} MB')
            parts.append(sample)
    if not parts: return None
    out = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    print(f'Combined: {out.shape} | {mem_mb(out):.0f} MB')
    return out


def make_synthetic(n_users=5000, n_movies=1000, n_ratings=300_000, seed=RANDOM_SEED):
    print(f'Generating synthetic data ({n_ratings:,} ratings)...')
    rng  = np.random.default_rng(seed)
    uw   = rng.exponential(1.0, n_users);  uw  /= uw.sum()
    mw   = rng.exponential(1.0, n_movies); mw  /= mw.sum()
    users  = rng.choice(np.arange(1, n_users+1,  dtype='int32'),  n_ratings, p=uw)
    movies = rng.choice(np.arange(1, n_movies+1, dtype='int16'),  n_ratings, p=mw)
    ub = rng.normal(0, 0.5, n_users+1)
    mb = rng.normal(0, 0.5, n_movies+1)
    raw = 3.0 + ub[users] + mb[movies] + rng.normal(0, 0.5, n_ratings)
    ratings = np.clip(np.round(raw), 1, 5).astype('int8')
    base = pd.Timestamp('1999-01-01')
    days = rng.integers(0, 365*5, n_ratings)
    dates = [(base + pd.Timedelta(d,'d')).strftime('%Y-%m-%d') for d in days]
    df = pd.DataFrame({'user_id':users,'movie_id':movies,'rating':ratings,'date':dates})
    df = df.drop_duplicates(['user_id','movie_id']).reset_index(drop=True)
    print(f'Synthetic: {df.shape} | {mem_mb(df):.0f} MB')
    return df


# ── Load / cache ──────────────────────────────────────────────────────────────
DATA_SOURCE = 'unknown'

if os.path.exists(PARQUET_PATH):
    print(f'Loading cached parquet...')
    with mem_snapshot('parquet load'):
        ratings_df = pd.read_parquet(PARQUET_PATH)
    DATA_SOURCE = 'parquet_cache'
else:
    has_raw = any(os.path.exists(os.path.join(RAW_DATA_DIR, f'combined_data_{i}.txt'))
                  for i in range(1,5))
    if has_raw:
        print('Netflix raw files found — loading...')
        ratings_df = load_and_sample(RAW_DATA_DIR)
        DATA_SOURCE = 'netflix_real'
    else:
        print('Raw files not found — using SYNTHETIC fallback.')
        ratings_df = make_synthetic()
        DATA_SOURCE = 'synthetic'
    ratings_df.to_parquet(PARQUET_PATH, index=False)
    print(f'Cached: {PARQUET_PATH}')

print(f'\nSource : {DATA_SOURCE}')
print(f'Shape  : {ratings_df.shape}')
print(f'RAM RSS: {_rss_mb():.0f} MB')


## 4  Filtering, Encoding & Train/Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder

ram_guard(need_mb=1200, label='preprocessing')

print(f'Before: {len(ratings_df):,} rows')

uc = ratings_df['user_id'].value_counts()
ratings_df = ratings_df[ratings_df['user_id'].isin(uc[uc >= MIN_USER_RATINGS].index)]
del uc; gc.collect()
print(f'After user filter  (>={MIN_USER_RATINGS}): {len(ratings_df):,}')

mc = ratings_df['movie_id'].value_counts()
ratings_df = ratings_df[ratings_df['movie_id'].isin(mc[mc >= MIN_MOVIE_RATINGS].index)]
del mc; gc.collect()
print(f'After movie filter (>={MIN_MOVIE_RATINGS}): {len(ratings_df):,}')

with mem_snapshot('date parse'):
    ratings_df['date'] = pd.to_datetime(ratings_df['date'], format='%Y-%m-%d')

user_enc = LabelEncoder()
item_enc = LabelEncoder()
ratings_df['user_idx'] = user_enc.fit_transform(ratings_df['user_id']).astype(np.int32)
ratings_df['item_idx'] = item_enc.fit_transform(ratings_df['movie_id']).astype(np.int32)

N_USERS  = len(user_enc.classes_)
N_ITEMS  = len(item_enc.classes_)
sparsity = 1.0 - len(ratings_df) / (N_USERS * N_ITEMS)
print(f'Users: {N_USERS:,}  Movies: {N_ITEMS:,}  Ratings: {len(ratings_df):,}')
print(f'Sparsity: {sparsity*100:.4f}%  RAM: {_rss_mb():.0f} MB')

# Estimate VRAM needed for NCF embeddings
embed_dim   = NCF_PARAMS['embedding_dim']
vram_needed = 4 * embed_dim * (N_USERS + N_ITEMS) * 4 / 1024**2  # 4 embedding tables
print(f'\nEstimated VRAM for embeddings: {vram_needed:.0f} MB')


In [ ]:
# Memory-efficient temporal split
with mem_snapshot('temporal split'):
    dfs   = ratings_df.sort_values(['user_id', 'date'])
    dfs['cc'] = dfs.groupby('user_id').cumcount()
    tot   = dfs.groupby('user_id')['user_id'].transform('count')
    cut   = (tot * (1 - TEST_SIZE)).astype(int)
    mask  = dfs['cc'] >= cut
    train_df = dfs[~mask].drop(columns=['cc']).reset_index(drop=True)
    test_df  = dfs[ mask].drop(columns=['cc']).reset_index(drop=True)
    del dfs, tot, cut, mask; gc.collect()

print(f'Train: {len(train_df):,}  Test: {len(test_df):,}')
print(f'Users in both: {len(set(train_df["user_id"]) & set(test_df["user_id"])):,}')
print(f'RAM: {_rss_mb():.0f} MB')


## 5  Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ratings_df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution'); axes[0].set_xlabel('Rating')

uc = ratings_df.groupby('user_id').size()
axes[1].hist(uc, bins=50, color='coral', log=True, edgecolor='white')
axes[1].set_title('Ratings per User (log)')

mc = ratings_df.groupby('movie_id').size()
axes[2].hist(mc, bins=50, color='mediumseagreen', log=True, edgecolor='white')
axes[2].set_title('Ratings per Movie (log)')

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'local_eda.png'), dpi=100)
plt.show()
del uc, mc; gc.collect()

mem_col = ratings_df.memory_usage(deep=True).drop('Index') / 1024**2
print(f'DataFrame RAM: {mem_col.sum():.1f} MB')
print(mem_col.sort_values(ascending=False).to_string())


## 6  SVD

> KNN is skipped locally — the similarity matrix for 5M rows can require 3-4 GB RAM
> and takes 20+ minutes on CPU. Run KNN on Kaggle.


In [ ]:
ram_guard(need_mb=1000, label='SVD training')

from surprise import Reader, Dataset, SVD

reader = Reader(rating_scale=(1, 5))
with mem_snapshot('Surprise dataset'):
    s_data     = Dataset.load_from_df(train_df[['user_id','movie_id','rating']], reader)
    s_trainset = s_data.build_full_trainset()
print(f'Trainset: {s_trainset.n_users:,} users  {s_trainset.n_items:,} items')

with mem_snapshot('SVD fit'):
    svd_model = SVD(**SVD_PARAMS, random_state=RANDOM_SEED)
    svd_model.fit(s_trainset)

print(f'SVD trained  |  RAM: {_rss_mb():.0f} MB')


## 7  Neural Collaborative Filtering (NeuMF) — GPU Accelerated

**RTX 3050 6 GB optimisations**:
- `batch_size=1024` — balanced for laptop GPU thermals.
- `pin_memory=True` + `num_workers=0` — fastest on Windows without fork overhead.
- VRAM guard before training (checks free VRAM > 500 MB).
- `torch.cuda.empty_cache()` every epoch — returns allocator cache to CUDA pool.
- Validation in 4096-row mini-batches — prevents VRAM spike.
- `torch.backends.cudnn.benchmark = True` — auto-selects fastest conv algorithm.


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset as TDS, DataLoader

vram_guard(need_mb=500, label='NCF model init')

# Optimal for laptop GPU: benchmark finds fastest kernels after a few warmup batches
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True


class RatingsDataset(TDS):
    def __init__(self, df):
        self.u = df['user_idx'].values.astype(np.int32)
        self.i = df['item_idx'].values.astype(np.int32)
        self.r = df['rating'].values.astype(np.float32)
    def __len__(self): return len(self.r)
    def __getitem__(self, idx):
        return (torch.tensor(self.u[idx], dtype=torch.long),
                torch.tensor(self.i[idx], dtype=torch.long),
                torch.tensor(self.r[idx], dtype=torch.float32))


class NeuMF(nn.Module):
    """Neural Matrix Factorization (He et al. 2017)."""
    def __init__(self, n_u, n_i, d=64, h=[128,64,32], drop=0.2):
        super().__init__()
        self.ug = nn.Embedding(n_u, d); self.ig = nn.Embedding(n_i, d)
        self.um = nn.Embedding(n_u, d); self.im = nn.Embedding(n_i, d)
        layers, dim = [], d*2
        for hd in h:
            layers += [nn.Linear(dim, hd), nn.ReLU(), nn.Dropout(drop)]; dim=hd
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(d+h[-1], 1)
    def forward(self, u, i):
        gmf = self.ug(u) * self.ig(i)
        mlp = self.mlp(torch.cat([self.um(u), self.im(i)], -1))
        return self.out(torch.cat([gmf, mlp], -1)).squeeze(-1)


ncf = NeuMF(N_USERS, N_ITEMS,
            d=NCF_PARAMS['embedding_dim'],
            h=NCF_PARAMS['hidden_layers'],
            drop=NCF_PARAMS['dropout']).to(DEVICE)

n_p = sum(p.numel() for p in ncf.parameters() if p.requires_grad)
print(f'NeuMF params : {n_p:,}')
print(f'Device       : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'VRAM used    : {torch.cuda.memory_allocated(0)/1024**2:.0f} MB')
print(f'RAM RSS      : {_rss_mb():.0f} MB')


In [ ]:
# DataLoader — pin_memory speeds up CPU->GPU transfer; num_workers=0 avoids
# Windows fork crashes with CUDA contexts in child processes.
PIN = DEVICE.type == 'cuda'
train_ds     = RatingsDataset(train_df)
val_ds       = RatingsDataset(test_df)
train_loader = DataLoader(train_ds,
                          batch_size=NCF_PARAMS['batch_size'],
                          shuffle=True,
                          pin_memory=PIN,
                          num_workers=0)   # 0 is safest on Windows

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(ncf.parameters(), lr=NCF_PARAMS['lr'])
SAVE_PATH = os.path.join(PROCESSED_DIR, 'local_ncf.pt')

best_val, pat = float('inf'), 0
t_losses, v_rmses = [], []

print(f'Training NCF on {DEVICE} ...')
for epoch in range(NCF_PARAMS['epochs']):
    # ── train ──────────────────────────────────────────────────────────────────
    ncf.train(); total = 0.0
    for b_idx, (u, i, r) in enumerate(train_loader):
        u, i, r = u.to(DEVICE), i.to(DEVICE), r.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ncf(u, i), r)
        loss.backward()
        optimizer.step()
        total += loss.item()
        if b_idx % 100 == 0:
            vr = torch.cuda.memory_allocated(0)/1024**2 if PIN else 0
            print(f'  ep{epoch+1} b{b_idx}/{len(train_loader)}'
                  f'  loss={loss.item():.4f}  VRAM={vr:.0f}MB  RAM={_rss_mb():.0f}MB',
                  end='\r')
    avg = total / len(train_loader); t_losses.append(avg); print()

    # ── validate in 4096-row mini-batches (prevents VRAM spike) ───────────────
    ncf.eval(); preds, tgts = [], []
    with torch.no_grad():
        for vu, vi, vr in DataLoader(val_ds, batch_size=4096, num_workers=0):
            preds.extend(ncf(vu.to(DEVICE), vi.to(DEVICE)).cpu().numpy())
            tgts.extend(vr.numpy())
    rmse = float(np.sqrt(np.mean((np.array(preds)-np.array(tgts))**2)))
    v_rmses.append(rmse)

    if PIN: torch.cuda.empty_cache()   # return allocator cache to CUDA pool

    vram_str = f'  VRAM={torch.cuda.memory_allocated(0)/1024**2:.0f}MB' if PIN else ''
    print(f'Epoch {epoch+1:02d}  loss={avg:.4f}  val_rmse={rmse:.4f}'
          f'  RAM={_rss_mb():.0f}MB{vram_str}')

    if rmse < best_val:
        best_val, pat = rmse, 0
        torch.save(ncf.state_dict(), SAVE_PATH)
    else:
        pat += 1
        if pat >= NCF_PARAMS['patience']:
            print(f'Early stopping at epoch {epoch+1}'); break

ncf.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
print(f'\nBest Val RMSE : {best_val:.4f}')
if PIN:
    print(f'Peak VRAM used: check nvidia-smi during training for thermal throttle')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t_losses, 'o-', color='steelblue', label='Train Loss')
axes[0].set_title('NCF Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[1].plot(v_rmses,  's-', color='coral', label='Val RMSE')
axes[1].set_title('NCF Validation RMSE'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RMSE')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'local_ncf_curves.png'), dpi=100)
plt.show()


## 8  Evaluation

In [ ]:
from surprise import accuracy as sa

def eval_surprise(model, df, name, max_r=50_000):
    df_ev = df.sample(min(max_r, len(df)), random_state=RANDOM_SEED)
    preds = [model.predict(str(r.user_id), str(r.movie_id), r_ui=float(r.rating))
             for r in df_ev[['user_id','movie_id','rating']].itertuples(index=False)]
    rmse = sa.rmse(preds, verbose=False)
    mae  = sa.mae( preds, verbose=False)
    print(f'{name:<20} RMSE={rmse:.4f}  MAE={mae:.4f}')
    return rmse, mae

def eval_ncf_gpu(model, df, device=DEVICE):
    model.eval(); p, t = [], []
    with torch.no_grad():
        for u,i,r in DataLoader(RatingsDataset(df), batch_size=4096, num_workers=0):
            p.extend(model(u.to(device), i.to(device)).cpu().numpy())
            t.extend(r.numpy())
    p = np.clip(np.array(p),1,5); t = np.array(t)
    rmse = float(np.sqrt(np.mean((p-t)**2)))
    mae  = float(np.mean(np.abs(p-t)))
    print(f'{"NeuMF-NCF":<20} RMSE={rmse:.4f}  MAE={mae:.4f}')
    return rmse, mae

print('='*48)
print('Evaluation on full test set')
print('='*48)
sv_r, sv_m = eval_surprise(svd_model, test_df, 'SVD')
nc_r, nc_m = eval_ncf_gpu(ncf, test_df)
print(f'RAM: {_rss_mb():.0f} MB')


In [ ]:
results = pd.DataFrame([
    {'Model':'SVD',       'RMSE':sv_r, 'MAE':sv_m},
    {'Model':'NeuMF-NCF', 'RMSE':nc_r, 'MAE':nc_m},
]).set_index('Model').round(4)
print(results.to_string())

clrs = ['steelblue','mediumpurple']
fig, ax = plt.subplots(1,2,figsize=(9,4))
results['RMSE'].plot(kind='bar',ax=ax[0],color=clrs,edgecolor='white')
ax[0].set_title('RMSE (lower=better)'); ax[0].tick_params(axis='x',rotation=10)
results['MAE'].plot(kind='bar', ax=ax[1],color=clrs,edgecolor='white')
ax[1].set_title('MAE (lower=better)');  ax[1].tick_params(axis='x',rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'local_model_comparison.png'), dpi=100)
plt.show()


## 9  Sample Recommendations

In [ ]:
# Load movie titles
TITLES_PATH = None
for tp in [os.path.join(RAW_DATA_DIR,'movie_titles.csv'),
           os.path.join(RAW_DATA_DIR,'movie_titles.txt')]:
    if os.path.exists(tp): TITLES_PATH=tp; break

titles_df = None
if TITLES_PATH:
    recs=[]
    with open(TITLES_PATH,'r',encoding='ISO-8859-1') as f:
        for line in f:
            p=line.strip().split(',',2)
            if len(p)==3: recs.append({'movie_id':int(p[0]),'title':p[2].strip()})
    titles_df=pd.DataFrame(recs)
    print(f'Loaded {len(titles_df)} titles')
else:
    print('movie_titles not found')

def get_title(mid):
    if titles_df is None: return str(mid)
    r = titles_df[titles_df['movie_id']==mid]
    return r['title'].values[0] if len(r) else str(mid)

rng         = np.random.default_rng(RANDOM_SEED)
sample_user = int(rng.choice(test_df['user_id'].unique()))
rated       = set(ratings_df[ratings_df['user_id']==sample_user]['movie_id'])
all_movies  = list(ratings_df['movie_id'].unique())
print(f'User {sample_user}  |  rated {len(rated)} movies')

svd_recs = sorted(
    [(m, svd_model.predict(str(sample_user),str(m)).est) for m in all_movies if m not in rated],
    key=lambda x:x[1], reverse=True)
print(f'\nTop-{TOP_K} SVD:')
for rank,(mid,sc) in enumerate(svd_recs[:TOP_K],1):
    print(f'  {rank:2d}. {get_title(mid):<45} (est={sc:.2f})')


In [ ]:
# NeuMF recs — GPU batched
cands=[m for m in all_movies if m not in rated]
try:
    u_idx=int(user_enc.transform([sample_user])[0])
    ci,cm=[],[]
    for m in cands:
        try: ci.append(int(item_enc.transform([m])[0])); cm.append(m)
        except ValueError: pass
    ncf.eval(); scores=[]; BS=1024
    with torch.no_grad():
        for s in range(0,len(ci),BS):
            i_t=torch.tensor(ci[s:s+BS],dtype=torch.long).to(DEVICE)
            u_t=torch.tensor([u_idx]*len(i_t),dtype=torch.long).to(DEVICE)
            scores.extend(ncf(u_t,i_t).cpu().numpy())
    ncf_recs=sorted(zip(cm,scores),key=lambda x:x[1],reverse=True)
    print(f'Top-{TOP_K} NeuMF:')
    for rank,(mid,sc) in enumerate(ncf_recs[:TOP_K],1):
        print(f'  {rank:2d}. {get_title(mid):<45} (est={sc:.2f})')
except Exception as e:
    print(f'NeuMF recs error: {e}')


## 10  Full System Diagnostic

In [ ]:
print('='*55)
print('  SYSTEM DIAGNOSTIC')
print('='*55)
if _HAS_PSUTIL:
    mi = psutil.Process(os.getpid()).memory_info()
    vm = psutil.virtual_memory()
    print(f'  CPU cores        : {os.cpu_count()}')
    print(f'  Process RSS      : {mi.rss/1024**3:.2f} GB')
    print(f'  System total RAM : {vm.total/1024**3:.1f} GB')
    print(f'  System available : {vm.available/1024**3:.1f} GB ({vm.percent:.0f}% used)')
    if vm.percent > 85:
        print('  *** WARNING: >85% RAM used — consider restarting kernel')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    alloc = torch.cuda.memory_allocated(0)/1024**2
    resvd = torch.cuda.memory_reserved(0)/1024**2
    total = props.total_memory/1024**2
    print(f'  GPU              : {props.name}')
    print(f'  VRAM total       : {total:.0f} MB')
    print(f'  VRAM allocated   : {alloc:.0f} MB')
    print(f'  VRAM reserved    : {resvd:.0f} MB')
    print(f'  VRAM free        : {total-resvd:.0f} MB')
print('='*55)

print('\nTop 10 globals by size:')
objs={k:sys.getsizeof(v)/1024**2 for k,v in globals().items()
      if not k.startswith('_') and sys.getsizeof(v) > 0.1}
for nm,sz in sorted(objs.items(),key=lambda x:x[1],reverse=True)[:10]:
    print(f'  {nm:<28} {sz:>7.1f} MB')


## 11  Save & Cleanup

In [ ]:
import pickle

enc_path = os.path.join(PROCESSED_DIR,'local_encoders.pkl')
with open(enc_path,'wb') as f:
    pickle.dump({'user_enc':user_enc,'item_enc':item_enc},f)
print(f'Encoders -> {enc_path}')

results.to_csv(os.path.join(PROCESSED_DIR,'local_results.csv'))
print(f'Results  -> {PROCESSED_DIR}/local_results.csv')
print(f'NCF      -> {SAVE_PATH}')

# Release training dataset objects
del train_ds, val_ds, train_loader
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print(f'\nFinal RAM  : {_rss_mb():.0f} MB')
if DEVICE.type == 'cuda':
    print(f'Final VRAM : {torch.cuda.memory_allocated(0)/1024**2:.0f} MB allocated')

print('\nOutput files:')
for fn in sorted(os.listdir(PROCESSED_DIR)):
    sz=os.path.getsize(os.path.join(PROCESSED_DIR,fn))/1024**2
    print(f'  {fn:<45} {sz:.1f} MB')
print('\nAll done!')
